# Vectorized Preference

Format:
```
{
  [
    { "prompt": prompt1,
      "chosen": str,
      "rejected": str
    },
    { "prompt": prompt2,
      "chosen": str,
      "rejected": str
    },
    ...
  ]
}

```

In [ ]:
!source .env

: 

In [ ]:
import json
from datasets import load_dataset
import os

folder_path = os.getenv("DATA_PATH")

def generate_preference_dataset(
    input_dataset_name="openbmb/UltraFeedback",
    output_filename="ultrafeedback_weighted.jsonl",
    weights=None
):
    print(f"Loading dataset: {input_dataset_name}...")
    ds = load_dataset(input_dataset_name, split="train")

    print("Processing rows and calculating scores...")

    output_data = []

    for i, row in enumerate(ds):
        instruction = row['instruction']
        completions = row['completions']

        best_score = -float('inf')
        worst_score = float('inf')
        best_response = None
        worst_response = None

        for completion in completions:
            response_text = completion['response']

            annotations = completion.get('annotations', {})

            current_score = 0.0
            valid_metrics = 0

            for metric, weight in weights.items():
                metric_data = annotations[metric]

                rating_str = metric_data.get('Rating', '0')
                try:
                    rating = float(rating_str)
                except ValueError:
                    rating = 0.0

                current_score += rating * weight

            if current_score > best_score:
                best_score = current_score
                best_response = response_text

            if current_score < worst_score:
                worst_score = current_score
                worst_response = response_text

        if best_response and worst_response and best_score != worst_score:
            new_entry = {
                "prompt": instruction,
                "chosen": best_response,
                "rejected": worst_response,
            }
            output_data.append(new_entry)

        if (i + 1) % 10000 == 0:
            print(f"Processed {i + 1} rows...")

    print(f"Finished processing. Total valid pairs: {len(output_data)}")
    print(f"Saving to {output_filename}...")

    with open(folder_path + output_filename, 'w', encoding='utf-8') as f:
        for entry in output_data:
            json.dump(entry, f)
            f.write('\n')

    print("Done!")


avg_weights = {
    "instruction_following": 0.25,
    "honesty": 0.25,
    "truthfulness": 0.25,
    "helpfulness": 0.25
}

student_weights = {
    "instruction_following": 0.35,
    "honesty": 0.1,
    "truthfulness": 0.25,
    "helpfulness": 0.3
}

professor_weights = {
    "instruction_following": 0.2,
    "honesty": 0.25,
    "truthfulness": 0.45,
    "helpfulness": 0.1
}

swe_weights = {
    "instruction_following": 0.35,
    "honesty": 0.1,
    "truthfulness": 0.2,
    "helpfulness": 0.35
}


generate_preference_dataset(weights=student_weights, output_filename="ultrafeedback_student.jsonl")
generate_preference_dataset(weights=swe_weights, output_filename="ultrafeedback_professor.jsonl")
generate_preference_dataset(weights=swe_weights, output_filename="ultrafeedback_swe.jsonl")

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Loading dataset: openbmb/UltraFeedback...
Processing rows and calculating scores...
Processed 10000 rows...
Processed 20000 rows...
Processed 30000 rows...
Processed 40000 rows...
Processed 50000 rows...
Processed 60000 rows...
Finished processing. Total valid pairs: 63540
Saving to ultrafeedback_swe.jsonl...
Done!


# Traditional DPO Preference

Format:
```
{
  [
    { "prompt": prompt1,
      "responses": [
        {
          "metric": "instruction_following",
          "chosen": str,
          "rejected": str,
          "score_chosen": float,
          "score_rejected": float,
        },
        {
          "metric": "honesty",
          "chosen": str,
          "rejected": str,
          "score_chosen": float,
          "score_rejected": float,
        },
        {
          "metric": "truthfulness",
          "chosen": str,
          "rejected": str,
          "score_chosen": float,
          "score_rejected": float,
        },
        {
          "metric": "helpfulness",
          "chosen": str,
          "rejected": str,
          "score_chosen": float,
          "score_rejected": float,
        }
      ]
    },

    ...
  ]
}

```

In [23]:
def generate_metric_specific_dataset(
    input_dataset_name="openbmb/UltraFeedback",
    output_filename="ultrafeedback_metric_specific.jsonl"
):
    print(f"Loading dataset: {input_dataset_name}...")
    ds = load_dataset(input_dataset_name, split="train")

    # The four specific metrics we are interested in
    metrics = [
        "instruction_following",
        "honesty",
        "truthfulness",
        "helpfulness"
    ]

    output_data = []

    print("Processing rows by individual metric...")

    for i, row in enumerate(ds):
        instruction = row['instruction']
        completions = row['completions']

        responses_data = []

        for metric in metrics:
            candidates = []

            for completion in completions:
                try:
                    text = completion['response']
                    annotations = completion.get('annotations', {})

                    metric_data = annotations.get(metric, {})
                    rating_str = metric_data.get('Rating', '0')

                    try:
                        score = float(rating_str)
                    except ValueError:
                        score = 0.0

                    candidates.append({
                        "text": text,
                        "score": score
                    })
                except Exception:
                    continue

            if not candidates:
                continue

            candidates.sort(key=lambda x: x['score'], reverse=True)

            best_candidate = candidates[0]
            worst_candidate = candidates[-1]

            if best_candidate['score'] > worst_candidate['score']:
                metric_entry = {
                    "metric": metric,
                    "chosen": best_candidate['text'],
                    "rejected": worst_candidate['text'],
                    "score_chosen": best_candidate['score'],
                    "score_rejected": worst_candidate['score']
                }
                responses_data.append(metric_entry)

        if responses_data:
            new_row = {
                "prompt": instruction,
                "responses": responses_data
            }
            output_data.append(new_row)

        if (i + 1) % 10000 == 0:
            print(f"Processed {i + 1} rows...")

    print(f"Finished processing. Total valid rows: {len(output_data)}")
    print(f"Saving to {output_filename}...")

    with open(folder_path + output_filename, 'w', encoding='utf-8') as f:
        for entry in output_data:
            json.dump(entry, f)
            f.write('\n')


generate_metric_specific_dataset()

Loading dataset: openbmb/UltraFeedback...
Processing rows by individual metric...
Processed 10000 rows...
Processed 20000 rows...
Processed 30000 rows...
Processed 40000 rows...
Processed 50000 rows...
Processed 60000 rows...
Finished processing. Total valid rows: 63790
Saving to ultrafeedback_metric_specific.jsonl...
Done!


# Align Datasets

In [ ]:
def align_datasets(file_paths, suffix="_aligned"):
    """
    Aligns multiple JSONL datasets to ensure they contain the exact same instructions.

    Args:
        file_paths (list): List of filenames (e.g., ['student.jsonl', 'professor.jsonl'])
        suffix (str): Suffix to add to the new filenames.
    """

    prompt_sets = []

    print("Step 1: analyzing prompts in all files...")

    for fp in file_paths:
        current_prompts = set()
        try:
            with open(fp, 'r', encoding='utf-8') as f:
                for line in f:
                    data = json.loads(line)
                    if 'prompt' in data:
                        current_prompts.add(data['prompt'])

            prompt_sets.append(current_prompts)
            print(f"   - {fp}: Found {len(current_prompts)} unique prompts.")

        except FileNotFoundError:
            print(f"Error: Could not find file {fp}")
            return

    if not prompt_sets:
        return

    common_prompts = set.intersection(*prompt_sets)
    print(f"\nAlignment Complete. Common intersection count: {len(common_prompts)} instructions.")
    print("-" * 40)

    print("Step 2: Writing aligned files...")

    for fp in file_paths:
        new_filename = fp.replace('.jsonl', f'{suffix}.jsonl')

        seen_prompts = set()
        count = 0

        with open(fp, 'r', encoding='utf-8') as infile, \
             open(new_filename, 'w', encoding='utf-8') as outfile:

            for line in infile:
                data = json.loads(line)
                prompt = data.get('prompt')

                if prompt in common_prompts and prompt not in seen_prompts:
                    json.dump(data, outfile)
                    outfile.write('\n')
                    seen_prompts.add(prompt)
                    count += 1

        print(f"   - Saved {new_filename} ({count} rows)")

In [38]:
my_files = [
    folder_path + "ultrafeedback_student.jsonl",
    folder_path + "ultrafeedback_professor.jsonl",
    folder_path + "ultrafeedback_swe.jsonl",
    folder_path + "ultrafeedback_metric_specific.jsonl"
]

try:
    align_datasets(my_files)
except Exception as e:
    print(f"An error occurred: {e}")
    print("Did you make sure to update 'my_files' with your actual filenames?")

Step 1: analyzing prompts in all files...
   - /content/gdrive/My Drive/ultrafeedback_student.jsonl: Found 63586 unique prompts.
   - /content/gdrive/My Drive/ultrafeedback_professor.jsonl: Found 63589 unique prompts.
   - /content/gdrive/My Drive/ultrafeedback_swe.jsonl: Found 63511 unique prompts.
   - /content/gdrive/My Drive/ultrafeedback_metric_specific.jsonl: Found 63760 unique prompts.

Alignment Complete. Common intersection count: 63502 instructions.
----------------------------------------
Step 2: Writing aligned files...
   - Saved /content/gdrive/My Drive/ultrafeedback_student_aligned.jsonl (63502 rows)
   - Saved /content/gdrive/My Drive/ultrafeedback_professor_aligned.jsonl (63502 rows)
   - Saved /content/gdrive/My Drive/ultrafeedback_swe_aligned.jsonl (63502 rows)
   - Saved /content/gdrive/My Drive/ultrafeedback_metric_specific_aligned.jsonl (63502 rows)
